In [1]:
# importing necessary libraries for building and training the ANN model
import tensorflow as tf
from tensorflow.keras.models import load_model
import pickle
import pandas as pd
import numpy as np

In [2]:
# Load the trained model, scaler pickle,onehot
model=load_model('model.h5')

# load the encoder and scaler
with open('onehot_encoder_geo.pkl','rb') as file:
    label_encoder_geo=pickle.load(file)

with open('label_encoder_gender.pkl', 'rb') as file:
    label_encoder_gender = pickle.load(file)

with open('scaler.pkl', 'rb') as file:
    scaler = pickle.load(file)

In [3]:
# Example input data
input_data = {
    'CreditScore': 600,
    'Geography': 'France',
    'Gender': 'Male',
    'Age': 40,
    'Tenure': 3,
    'Balance': 60000,
    'NumOfProducts': 2,
    'HasCrCard': 1,
    'IsActiveMember': 1,
    'EstimatedSalary': 50000
}

In [ ]:
# One-hot encode Geography

# Get the Geography value and convert it into a one-hot encoded format
# [[...]] makes it a 2D input as required by the encoder
# .toarray() converts the sparse output into a regular NumPy array
geo_encoded = label_encoder_geo.transform([[input_data['Geography']]]).toarray()

# Create a DataFrame from the encoded array
# get_feature_names_out() generates column names for each geography category
# labe
geo_encoded_df = pd.DataFrame(
    geo_encoded,
    columns=label_encoder_geo.get_feature_names_out(['Geography'])
)

geo_encoded_df

c:\CODING\Machine Learning\ML\DL\DL practical\venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


In [5]:
input_df=pd.DataFrame([input_data])
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,Male,40,3,60000,2,1,1,50000


In [6]:
## Encode categorical variables
input_df['Gender']=label_encoder_gender.transform(input_df['Gender'])
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,1,40,3,60000,2,1,1,50000


In [7]:
## concatination one hot encoded 
input_df=pd.concat([input_df.drop("Geography",axis=1),geo_encoded_df],axis=1)
input_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,1,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [8]:
## Scaling the input data
input_scaled=scaler.transform(input_df)
input_scaled

array([[-0.53598516,  0.91324755,  0.10479359, -0.69539349, -0.25781119,
         0.80843615,  0.64920267,  0.97481699, -0.87683221,  1.00150113,
        -0.57946723, -0.57638802]])

In [9]:
## PRedict churn
prediction=model.predict(input_scaled)
prediction

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 109ms/step


array([[0.0335623]], dtype=float32)

In [10]:
prediction_proba = prediction[0][0]

In [11]:
prediction_proba

0.033562295

In [12]:
if prediction_proba > 0.5:
    print('The customer is likely to churn.')
else:
    print('The customer is not likely to churn.')

The customer is not likely to churn.


In [ ]:
# ============================================================
# CUSTOMER CHURN PREDICTION - COMPLETE INFERENCE WORKFLOW
# ============================================================

# 1. Import required libraries
#    Why?
#    - TensorFlow/Keras is required to load and use the trained ANN model.
#    - Pickle is used to load saved preprocessing objects.
#    - Pandas helps organize and manipulate tabular data.
#    - NumPy supports numerical computations.

# 2. Load the trained ANN model
#    Why?
#    - The model has already learned patterns from historical customer data.
#    - We load it so we can make predictions on new customer records.

# 3. Load saved preprocessing objects
#    Why?
#    - During training, categorical features were encoded and numerical
#      features were scaled.
#    - New input data must undergo the exact same transformations;
#      otherwise, the model will receive data in a different format and
#      predictions may become inaccurate.

# 4. Create input data for a customer
#    Why?
#    - This represents a new customer's information for which we want
#      to predict the likelihood of churn.

# 5. Apply One-Hot Encoding to Geography
#    Why?
#    - Machine learning models cannot directly understand text values
#      such as "France", "Germany", or "Spain".
#    - One-Hot Encoding converts each country into separate binary columns,
#      allowing the model to process geographical information numerically.

# 6. Convert input data into a DataFrame
#    Why?
#    - The preprocessing pipeline (encoders and scaler) expects data
#      in a structured tabular format.
#    - DataFrames make feature handling easier and more organized.

# 7. Apply Label Encoding to Gender
#    Why?
#    - The Gender feature is categorical.
#    - Label Encoding converts values like "Male" and "Female" into
#      numerical representations that the model can understand.

# 8. Combine encoded Geography features with remaining features
#    Why?
#    - After encoding, the original Geography column is no longer needed.
#    - The newly created geography columns must be added to the dataset
#      so the feature structure matches the training data.

# 9. Scale all input features using the trained scaler
#    Why?
#    - Features such as Salary, Balance, and Credit Score have very
#      different ranges.
#    - Scaling standardizes feature values and ensures the model receives
#      data in the same format used during training.
#    - This improves prediction consistency and performance.

# 10. Pass the processed data to the ANN model
#     Why?
#     - The model uses learned weights and relationships between features
#       to estimate the probability of customer churn.

# 11. Obtain the churn probability
#     Why?
#     - The ANN outputs a probability value between 0 and 1.
#     - Values closer to 1 indicate a higher likelihood of churn.

# 12. Apply a decision threshold (0.5)
#     Why?
#     - A probability alone may not be easy to interpret.
#     - Using a threshold converts the probability into a final business
#       decision:
#         > 0.5  → Likely to churn
#         ≤ 0.5 → Likely to stay

# 13. Display the final prediction
#     Why?
#     - Provides an easy-to-understand result for end users or business
#       stakeholders without requiring them to interpret probabilities.
# ============================================================